# Tuned LSTM Baseline

기존 제출 조건은 그대로 유지합니다.

- `train.jsonl`로 학습
- `test.jsonl`로 예측
- `submission.csv` 저장
- 제출 컬럼: `row_id`, `helpful_yes_pred`, `helpful_total_pred`, `overall_pred`
- 평가 기준: 각 target RMSE 계산 후 평균

변경한 핵심은 수치 튜닝이 아니라 baseline 구조 보강입니다.

1. train/valid split으로 내부 RMSE 확인
2. padding을 무시하는 BiLSTM 사용
3. review/summary 길이, 시간, 상품/리뷰어 빈도 feature 추가
4. target scaling 적용
5. final model은 전체 train으로 다시 학습 후 submission 생성
6. helpful/overall의 가능한 값 범위에 맞춰 후처리


In [ ]:
from huggingface_hub import hf_hub_download
import os
import shutil

repo_id = "hongdune/BenchMarkDataset"
repo_type = "dataset"

files = [
    "train.jsonl",
    "test.jsonl",
    "submission.csv"
]

save_dir = "."

for filename in files:
    if os.path.exists(os.path.join(save_dir, filename)):
        print(f"already exists: {filename}")
        continue

    path = hf_hub_download(
        repo_id=repo_id,
        filename=filename,
        repo_type=repo_type
    )

    shutil.copy(path, os.path.join(save_dir, filename))
    print(f"saved: {os.path.join(save_dir, filename)}")

In [ ]:
import ast
import copy
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from transformers import BertTokenizer

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# 기본 하이퍼파라미터
MAX_LEN_REVIEW = 256
MAX_LEN_SUMMARY = 24

BATCH_SIZE = 64
EPOCHS = 15
PATIENCE = 4

EMBED_DIM = 128
HIDDEN_DIM = 128
DROPOUT = 0.3

LR = 5e-4
WEIGHT_DECAY = 1e-4

TARGET_COLS = ["helpful_yes", "helpful_total", "overall"]

META_COLS = [
    "review_len",
    "summary_len",
    "review_char_len",
    "summary_char_len",
    "review_age_days",
    "asin_count",
    "reviewer_count",
]

In [ ]:
def parse_helpful(x):
    if isinstance(x, list) and len(x) == 2:
        return x[0], x[1]
    if isinstance(x, str):
        y = ast.literal_eval(x)
        return y[0], y[1]
    return 0, 0


def load_train(path="train.jsonl"):
    df = pd.read_json(path, lines=True)
    df["helpful_yes"], df["helpful_total"] = zip(*df["helpful"].apply(parse_helpful))
    return df


def load_test(path="test.jsonl"):
    df = pd.read_json(path, lines=True)
    if "row_id" not in df.columns:
        df = df.reset_index(drop=True)
        df["row_id"] = df.index.astype(int)
    return df


raw_df = load_train("train.jsonl")
raw_df.head()

In [ ]:
def fit_feature_reference(df):
    ref = {}
    ref["asin_counts"] = df["asin"].fillna("UNKNOWN").value_counts().to_dict()
    ref["reviewer_counts"] = df["reviewerID"].fillna("UNKNOWN").value_counts().to_dict()
    ref["max_unix_time"] = df["unixReviewTime"].fillna(df["unixReviewTime"].median()).max()
    return ref


def add_features(df, ref):
    df = df.copy()

    df["reviewText"] = df["reviewText"].fillna("")
    df["summary"] = df["summary"].fillna("")
    df["asin"] = df["asin"].fillna("UNKNOWN")
    df["reviewerID"] = df["reviewerID"].fillna("UNKNOWN")

    df["review_len"] = df["reviewText"].apply(lambda x: len(x.split()))
    df["summary_len"] = df["summary"].apply(lambda x: len(x.split()))
    df["review_char_len"] = df["reviewText"].apply(len)
    df["summary_char_len"] = df["summary"].apply(len)

    df["unixReviewTime"] = df["unixReviewTime"].fillna(df["unixReviewTime"].median())
    df["review_age_days"] = (ref["max_unix_time"] - df["unixReviewTime"]) / (60 * 60 * 24)

    df["asin_count"] = df["asin"].map(ref["asin_counts"]).fillna(0)
    df["reviewer_count"] = df["reviewerID"].map(ref["reviewer_counts"]).fillna(0)

    return df


# 내부 검증용 split
train_raw, valid_raw = train_test_split(
    raw_df,
    test_size=0.2,
    random_state=SEED,
    stratify=raw_df["overall"].astype(int)
)

feature_ref = fit_feature_reference(train_raw)

train_df = add_features(train_raw, feature_ref).reset_index(drop=True)
valid_df = add_features(valid_raw, feature_ref).reset_index(drop=True)

print("train:", train_df.shape)
print("valid:", valid_df.shape)
train_df[META_COLS + TARGET_COLS].head()

In [ ]:
# target과 meta feature의 스케일을 맞춘다.
# 원본 target은 helpful 계열과 overall의 범위가 너무 다르므로 그대로 MSE를 걸면 helpful 쪽으로 loss가 쏠린다.

meta_scaler = StandardScaler()
target_scaler = StandardScaler()

train_df[META_COLS] = meta_scaler.fit_transform(train_df[META_COLS])
valid_df[META_COLS] = meta_scaler.transform(valid_df[META_COLS])

train_df[TARGET_COLS] = target_scaler.fit_transform(train_df[TARGET_COLS])
valid_df[TARGET_COLS] = target_scaler.transform(valid_df[TARGET_COLS])

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")


def tokenize_texts(texts, max_length):
    encoded = tokenizer(
        texts.tolist(),
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors=None,
    )
    return encoded["input_ids"]


train_df["reviewText_token"] = tokenize_texts(train_df["reviewText"], MAX_LEN_REVIEW)
train_df["summary_token"] = tokenize_texts(train_df["summary"], MAX_LEN_SUMMARY)

valid_df["reviewText_token"] = tokenize_texts(valid_df["reviewText"], MAX_LEN_REVIEW)
valid_df["summary_token"] = tokenize_texts(valid_df["summary"], MAX_LEN_SUMMARY)

print(len(train_df["reviewText_token"].iloc[0]), len(train_df["summary_token"].iloc[0]))

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, df, meta_cols, target_cols=None):
        self.review_tokens = np.array(df["reviewText_token"].tolist(), dtype=np.int64)
        self.summary_tokens = np.array(df["summary_token"].tolist(), dtype=np.int64)
        self.meta = df[meta_cols].values.astype(np.float32)

        self.has_target = target_cols is not None
        if self.has_target:
            self.target = df[target_cols].values.astype(np.float32)

    def __len__(self):
        return len(self.review_tokens)

    def __getitem__(self, idx):
        item = {
            "review_tokens": torch.tensor(self.review_tokens[idx], dtype=torch.long),
            "summary_tokens": torch.tensor(self.summary_tokens[idx], dtype=torch.long),
            "meta": torch.tensor(self.meta[idx], dtype=torch.float),
        }

        if self.has_target:
            item["target"] = torch.tensor(self.target[idx], dtype=torch.float)

        return item


train_dataset = ReviewDataset(train_df, META_COLS, TARGET_COLS)
valid_dataset = ReviewDataset(valid_df, META_COLS, TARGET_COLS)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
for key, value in batch.items():
    print(key, value.shape)

In [ ]:
class LSTMRegressor(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, meta_dim, dropout=0.3):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0
        )

        self.review_lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.summary_lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        # review BiLSTM: hidden_dim * 2
        # summary BiLSTM: hidden_dim * 2
        # meta: meta_dim
        input_dim = hidden_dim * 4 + meta_dim

        self.shared = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        self.helpful_head = nn.Linear(128, 2)  # helpful_yes, helpful_total
        self.overall_head = nn.Linear(128, 1)  # overall

    def encode_sequence(self, tokens, lstm):
        embedded = self.embedding(tokens)

        lengths = (tokens != 0).sum(dim=1).clamp(min=1).cpu()

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths,
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, _) = lstm(packed)

        # bidirectional=True, num_layers=1 기준
        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]
        encoded = torch.cat([forward_hidden, backward_hidden], dim=1)

        return encoded

    def forward(self, review_tokens, summary_tokens, meta):
        review_vec = self.encode_sequence(review_tokens, self.review_lstm)
        summary_vec = self.encode_sequence(summary_tokens, self.summary_lstm)

        x = torch.cat([review_vec, summary_vec, meta], dim=1)
        x = self.shared(x)

        helpful_pred = self.helpful_head(x)
        overall_pred = self.overall_head(x)

        # TARGET_COLS 순서와 맞춘다: helpful_yes, helpful_total, overall
        return torch.cat([helpful_pred, overall_pred], dim=1)

In [ ]:
def postprocess_predictions(pred):
    pred = pred.copy()

    # pred columns: helpful_yes, helpful_total, overall
    pred[:, 0] = np.clip(pred[:, 0], 0, None)
    pred[:, 1] = np.clip(pred[:, 1], 0, None)

    # helpful_yes는 helpful_total보다 클 수 없다.
    pred[:, 0] = np.minimum(pred[:, 0], pred[:, 1])

    # overall은 1~5 사이 점수다.
    pred[:, 2] = np.clip(pred[:, 2], 1, 5)

    return pred


def calculate_rmse(y_true, y_pred):
    rmse_each = np.sqrt(np.mean((y_true - y_pred) ** 2, axis=0))
    rmse_mean = rmse_each.mean()
    return rmse_each, rmse_mean


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for batch in loader:
        review = batch["review_tokens"].to(device)
        summary = batch["summary_tokens"].to(device)
        meta = batch["meta"].to(device)
        target = batch["target"].to(device)

        optimizer.zero_grad()
        pred = model(review, summary, meta)
        loss = criterion(pred, target)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        total_loss += loss.item() * review.size(0)

    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device, target_scaler):
    model.eval()
    total_loss = 0.0
    preds = []
    trues = []

    with torch.no_grad():
        for batch in loader:
            review = batch["review_tokens"].to(device)
            summary = batch["summary_tokens"].to(device)
            meta = batch["meta"].to(device)
            target = batch["target"].to(device)

            pred = model(review, summary, meta)
            loss = criterion(pred, target)

            total_loss += loss.item() * review.size(0)
            preds.append(pred.cpu().numpy())
            trues.append(target.cpu().numpy())

    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)

    preds_raw = target_scaler.inverse_transform(preds)
    trues_raw = target_scaler.inverse_transform(trues)

    preds_raw = postprocess_predictions(preds_raw)

    rmse_each, rmse_mean = calculate_rmse(trues_raw, preds_raw)

    return total_loss / len(loader.dataset), rmse_each, rmse_mean


def build_model():
    model = LSTMRegressor(
        vocab_size=tokenizer.vocab_size,
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        meta_dim=len(META_COLS),
        dropout=DROPOUT
    )
    return model.to(device)

In [ ]:
model = build_model()
criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

best_rmse = float("inf")
best_epoch = 0
best_model_state = None
patience_count = 0

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    valid_loss, valid_rmse_each, valid_rmse_mean = evaluate(
        model, valid_loader, criterion, device, target_scaler
    )

    print(f"Epoch [{epoch}/{EPOCHS}]")
    print(f"Train Loss: {train_loss:.4f} | Valid Loss: {valid_loss:.4f}")
    print(
        "Valid RMSE | "
        f"helpful_yes: {valid_rmse_each[0]:.4f}, "
        f"helpful_total: {valid_rmse_each[1]:.4f}, "
        f"overall: {valid_rmse_each[2]:.4f}, "
        f"mean: {valid_rmse_mean:.4f}"
    )
    print("-" * 70)

    if valid_rmse_mean < best_rmse:
        best_rmse = valid_rmse_mean
        best_epoch = epoch
        best_model_state = copy.deepcopy(model.state_dict())
        patience_count = 0
        print("Best model updated")
        print("-" * 70)
    else:
        patience_count += 1

    if patience_count >= PATIENCE:
        print("Early stopping")
        break

model.load_state_dict(best_model_state)
print("Best valid RMSE:", best_rmse)
print("Best epoch:", best_epoch)

## Final 학습

위 validation은 튜닝 판단용입니다. 제출용 모델은 같은 구조와 같은 하이퍼파라미터로 `train.jsonl` 전체를 다시 학습합니다.

`best_epoch`만큼만 학습해서 validation에서 좋았던 학습량을 그대로 사용합니다.


In [ ]:
# 전체 train 데이터 기준으로 feature/scaler/token을 다시 만든다.
final_feature_ref = fit_feature_reference(raw_df)
final_train_df = add_features(raw_df, final_feature_ref).reset_index(drop=True)

final_meta_scaler = StandardScaler()
final_target_scaler = StandardScaler()

final_train_df[META_COLS] = final_meta_scaler.fit_transform(final_train_df[META_COLS])
final_train_df[TARGET_COLS] = final_target_scaler.fit_transform(final_train_df[TARGET_COLS])

final_train_df["reviewText_token"] = tokenize_texts(final_train_df["reviewText"], MAX_LEN_REVIEW)
final_train_df["summary_token"] = tokenize_texts(final_train_df["summary"], MAX_LEN_SUMMARY)

final_train_dataset = ReviewDataset(final_train_df, META_COLS, TARGET_COLS)
final_train_loader = DataLoader(final_train_dataset, batch_size=BATCH_SIZE, shuffle=True)

final_model = build_model()
final_optimizer = optim.AdamW(final_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
final_criterion = nn.MSELoss()

final_epochs = max(1, best_epoch)

for epoch in range(1, final_epochs + 1):
    train_loss = train_one_epoch(
        final_model,
        final_train_loader,
        final_criterion,
        final_optimizer,
        device
    )
    print(f"Final Train Epoch [{epoch}/{final_epochs}] | Loss: {train_loss:.4f}")

In [ ]:
test_df = load_test("test.jsonl")
test_df = add_features(test_df, final_feature_ref).reset_index(drop=True)

test_df[META_COLS] = final_meta_scaler.transform(test_df[META_COLS])

test_df["reviewText_token"] = tokenize_texts(test_df["reviewText"], MAX_LEN_REVIEW)
test_df["summary_token"] = tokenize_texts(test_df["summary"], MAX_LEN_SUMMARY)

test_dataset = ReviewDataset(test_df, META_COLS, target_cols=None)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
final_model.eval()
all_preds = []

with torch.no_grad():
    for batch in test_loader:
        review = batch["review_tokens"].to(device)
        summary = batch["summary_tokens"].to(device)
        meta = batch["meta"].to(device)

        pred_scaled = final_model(review, summary, meta)
        all_preds.append(pred_scaled.cpu().numpy())

all_preds = np.concatenate(all_preds, axis=0)
all_preds = final_target_scaler.inverse_transform(all_preds)
all_preds = postprocess_predictions(all_preds)

pred_df = pd.DataFrame(
    all_preds,
    columns=["helpful_yes_pred", "helpful_total_pred", "overall_pred"]
)
pred_df["row_id"] = test_df["row_id"].values

pred_df.head()

In [ ]:
# sample submission의 row_id 순서를 유지해서 저장한다.
sample_submission = pd.read_csv("submission.csv")

submission = sample_submission[["row_id"]].merge(
    pred_df,
    on="row_id",
    how="left"
)

missing_count = submission[
    ["helpful_yes_pred", "helpful_total_pred", "overall_pred"]
].isna().any(axis=1).sum()

print("missing_count:", missing_count)

submission.to_csv("submission.csv", index=False)

print(submission.head())
print("shape:", submission.shape)

## 다음 튜닝 후보

위 코드가 정상적으로 돌아가면 다음 순서로 하나씩만 바꿔서 `Best valid RMSE`를 비교합니다.

1. `MAX_LEN_REVIEW`: 128 → 256 → 384
2. `HIDDEN_DIM`: 64 → 128 → 256
3. `LR`: 1e-3 → 5e-4 → 3e-4
4. `DROPOUT`: 0.2 → 0.3 → 0.5
5. `criterion`: `nn.MSELoss()` ↔ `nn.SmoothL1Loss()`

제출 조건은 마지막 cell에서 그대로 유지됩니다.
